#  ⭐ `dim_via_administracao`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root #, normalizar_via_administracao#, normalizar_via_fuzzy

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path)
bronze["VIA_ADMINISTRACAO"] = bronze["VIA_ADMINISTRACAO"].fillna("DESCONHECIDO")

bronze = pd.concat(
    [
        (
            bronze[["VIA_ADMINISTRACAO"]]
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
        (
            bronze[["VIA_ADMINISTRACAO_MAE_PAI"]].rename(columns={"VIA_ADMINISTRACAO_MAE_PAI": "VIA_ADMINISTRACAO"})
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
    ],
    ignore_index=True,
)

bronze.to_csv(
    Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_VIA_ADM.csv",
    sep=";",
    index=False,
)

In [4]:
bronze.head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,307235
1,oral,31059
2,infusão intravenosa,26376
3,intramuscular,19866
4,desconhecida,18764


In [5]:
bronze.VIA_ADMINISTRACAO.nunique()

1839

In [6]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [7]:
from utils.med_normalize_via_adm import normalizar_via_fuzzy


In [8]:
silver = pd.read_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";") ## Manual Normalization
 
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [9]:
silver["VIA_ADMINISTRACAO_CHAVE"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(txt, score_threshold=80))
)

silver["VIA_ADMINISTRACAO_VALOR"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(
        lambda txt: normalizar_via_fuzzy(
            txt,
            score_threshold=80,
            return_numeric=True,
        )
    )
)

silver["VIA_ADMINISTRACAO_DESCRIPTION"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(
        lambda txt: normalizar_via_fuzzy(
            txt,
            score_threshold=80,
            return_description=True,
        )
    )
)

In [10]:
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_VALOR', 'VIA_ADMINISTRACAO_DESCRIPTION'],
      dtype='object')

In [11]:
silver.head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION
0,DESCONHECIDO,307235,desconhecido,0,unknown
1,oral,31059,O,5,oral
2,infusão intravenosa,26376,P,6,parenteral
3,intramuscular,19866,P,6,parenteral
4,desconhecida,18764,desconhecido,0,unknown


In [12]:
(
    silver.groupby("VIA_ADMINISTRACAO_CHAVE")["FREQUENCIA_REGISTROS"]
    .sum()
    .reset_index(name="FREQUENCIA_REGISTROS_TOTAL")
).sort_values(by='FREQUENCIA_REGISTROS_TOTAL', ascending=False)

,VIA_ADMINISTRACAO_CHAVE,FREQUENCIA_REGISTROS_TOTAL
10,desconhecido,349186
5,P,144797
4,O,53614
1,Inhal,1929
8,TD,1417
2,Instill,672
0,Implant,556
3,N,323
7,SL,286
9,V,161


In [13]:
silver.query("VIA_ADMINISTRACAO_CHAVE == 1").head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION


In [14]:
hist = silver[['VIA_ADMINISTRACAO','VIA_ADMINISTRACAO_CHAVE','VIA_ADMINISTRACAO_VALOR', 'VIA_ADMINISTRACAO_DESCRIPTION']].sort_values(by='VIA_ADMINISTRACAO_VALOR')
hist.head()

,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION
0,DESCONHECIDO,desconhecido,0,unknown
1705,NORBERTP BATISTA,desconhecido,0,unknown
203,0108,desconhecido,0,unknown
847,"administrada por volta de 8h30, E2B R2 Code: 065 - Unknown",desconhecido,0,unknown
877,"braço direito, via desconhecida",desconhecido,0,unknown


In [15]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_via_adm/hist_via_adm.parquet", index=False)

# 🥇 Gold


In [16]:
dim = (
    hist[["VIA_ADMINISTRACAO_CHAVE", "VIA_ADMINISTRACAO_VALOR","VIA_ADMINISTRACAO_DESCRIPTION"]]
    .drop_duplicates()
    .sort_values(by="VIA_ADMINISTRACAO_VALOR")
) 
dim

,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION
0,desconhecido,0,unknown
97,Implant,1,implant
1310,Inhal,2,inhalation
468,Instill,3,instillation
152,N,4,nasal
1111,O,5,oral
1724,P,6,parenteral
1638,R,7,rectal
500,SL,8,sublingual/buccal/oromucosal
528,TD,9,transdermal


In [17]:
dim.to_csv(Path(project_root) / "data/03_gold/dim_via_adm/dim_via_adm.csv", sep=";", index=False)